# 03 · Pose-specific residue maps

Display pocket membership for individual poses. Chain and residue identifiers remain explicit. A nearby residue is not evidence of a verified binding interaction.

## Configuration

Use `config.local.json` as described in [the workflow guide](../docs/WORKFLOW.md). Inputs must be authorized for local analysis. Outputs are cleared in version control; no research result is asserted by this notebook.

In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'analysis/interactions.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Start Jupyter from this repository')
sys.path.insert(0, str(root))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from analysis.notebook_setup import load_config, load_manifest
config = load_config()
out = config['output_dir']


## Build residue-presence matrix

In [ ]:
pockets = pd.read_csv(out / 'pocket_residues.csv', dtype={'compound_id': str, 'pose_id': str})
required = {'compound_id', 'pose_id', 'chain', 'index'}
if not required <= set(pockets):
    raise ValueError(f'Pocket table requires {sorted(required)}')
pockets['residue_key'] = pockets['chain'].astype(str) + ':' + pockets['index'].astype(str)
matrix = pockets.assign(present=1).pivot_table(index=['compound_id', 'pose_id'],
    columns='residue_key', values='present', aggfunc='max', fill_value=0)
matrix.to_csv(out / 'pose_residue_presence.csv')
matrix.head()

## Visualize pose patterns

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(12, max(4, min(16, len(matrix) * .2))))
sns.heatmap(matrix, cmap=['#f5f8fb', '#246b9b'], cbar=False, ax=ax)
ax.set(title='Predicted pocket membership by compound and pose', xlabel='Chain:residue', ylabel='Compound / pose')
fig.tight_layout(); fig.savefig(out / 'pose_residue_heatmap.svg')
plt.show()